# Fase 13 — Productos científicos y de apoyo a decisiones

Esta fase convierte los productos validados de las fases 10–12 en tablas, figuras y reportes determinísticos.

Principios metodológicos:

- los hallazgos no se presentan como estudios independientes;
- calidad, transferibilidad, madurez operacional y estabilidad permanecen separadas;
- aceptación experta no equivale a efectividad;
- implementación o pilotaje no prueban resultados;
- la estabilidad analítica no justifica adopción automática;
- las recomendaciones sensibles se publican explícitamente con sus limitaciones.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from evidence_review.publication_products import (
    build_evidence_summary,
    build_final_recommendations,
    build_management_portfolio_table,
    build_product_manifest,
    build_publication_metrics,
    build_research_gap_table,
    build_source_influence_table,
    generate_publication_figures,
    load_publication_config,
    read_csv_robust,
    validate_publication_tables,
    write_publication_reports,
)

CONFIG_PATH = ROOT / 'config' / 'publication_products.yml'
config = load_publication_config(CONFIG_PATH, project_root=ROOT)
paths = config['paths']

print(f'Project root: {ROOT}')
print(f'Configuration: {CONFIG_PATH.relative_to(ROOT)}')
print(f'Output root: {paths["output_root"]}')


## 1. Cargar productos validados

La fase exige recomendaciones `accepted` o `corrected`, una clase de estabilidad por recomendación y cero problemas pendientes en la auditoría de robustez.


In [ ]:
required_inputs = {
    'matrix': 'synthesis_matrix_csv',
    'portfolio': 'management_portfolio_csv',
    'priorities': 'research_priority_csv',
    'recommendations': 'recommendation_validated_csv',
    'stability': 'recommendation_stability_csv',
    'source_influence': 'source_influence_csv',
    'source_concentration': 'source_concentration_csv',
    'independence': 'evidence_independence_csv',
    'robustness_issues': 'robustness_issues_csv',
}

frames, encodings = {}, {}
for name, key in required_inputs.items():
    input_path = ROOT / paths[key]
    if not input_path.exists():
        raise FileNotFoundError(input_path)
    frames[name], encodings[name] = read_csv_robust(input_path)
    print(f'{name}: {len(frames[name])} [{encodings[name]}]')

matrix = frames['matrix']
portfolio_input = frames['portfolio']
priorities_input = frames['priorities']
recommendations_input = frames['recommendations']
stability_input = frames['stability']
independence_input = frames['independence']
robustness_issues = frames['robustness_issues']

if matrix.empty:
    raise ValueError('The validated evidence synthesis matrix is empty.')
if recommendations_input.empty:
    raise ValueError('No expert-validated recommendations are available.')
if not robustness_issues.empty:
    raise ValueError('Phase 12 must have zero robustness issues before phase 13.')
invalid_review = recommendations_input.loc[
    ~recommendations_input['expert_review_status'].isin(['accepted', 'corrected'])
]
if not invalid_review.empty:
    raise ValueError('Recommendation input contains rows without expert validation.')

print(f'Validated findings: {len(matrix)}')
print(f'Independent sources: {matrix["source_id"].nunique()}')
print(f'Management options: {len(portfolio_input)}')
print(f'Validated recommendations: {len(recommendations_input)}')


## 2. Construir y validar tablas científicas


In [ ]:
evidence_summary = build_evidence_summary(matrix)
management_portfolio = build_management_portfolio_table(
    portfolio_input, stability_input
)
final_recommendations = build_final_recommendations(
    recommendations_input, stability_input, config
)
research_gaps = build_research_gap_table(
    priorities_input, final_recommendations
)
source_influence = build_source_influence_table(
    stability_input, independence_input
)
publication_metrics = build_publication_metrics(
    matrix, management_portfolio, final_recommendations, research_gaps
)

issues = validate_publication_tables(
    management_portfolio,
    final_recommendations,
    research_gaps,
    config,
    expected_recommendation_ids=list(
        recommendations_input['recommendation_id'].astype(str)
    ),
    robustness_issues=robustness_issues,
)

print(f'Publication-table issues: {len(issues)}')
display(evidence_summary)
display(management_portfolio)
display(final_recommendations)
display(publication_metrics)
display(issues.head(100))
if not issues.empty:
    raise ValueError('Resolve publication-table issues before export.')


## 3. Exportar tablas, figuras y reportes


In [ ]:
table_outputs = {
    'evidence_summary_csv': evidence_summary,
    'management_portfolio_table_csv': management_portfolio,
    'final_recommendations_csv': final_recommendations,
    'research_gaps_csv': research_gaps,
    'source_influence_table_csv': source_influence,
    'publication_metrics_csv': publication_metrics,
}
for key, frame in table_outputs.items():
    output_path = ROOT / paths[key]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f'{output_path.relative_to(ROOT)}: {len(frame)}')

figure_products = generate_publication_figures(
    matrix=matrix,
    portfolio=management_portfolio,
    final_recommendations=final_recommendations,
    source_table=source_influence,
    metrics=publication_metrics,
    root=ROOT,
    paths=paths,
    settings=config['figure_settings'],
)
report_products = write_publication_reports(
    evidence_summary=evidence_summary,
    portfolio=management_portfolio,
    final_recommendations=final_recommendations,
    research_gaps=research_gaps,
    source_table=source_influence,
    metrics=publication_metrics,
    root=ROOT,
    paths=paths,
    config=config,
)
display(figure_products)
display(report_products)


## 4. Manifiesto y validación final


In [ ]:
product_keys = [
    *table_outputs.keys(),
    'evidence_flow_png',
    'quality_transferability_png',
    'management_readiness_png',
    'recommendation_stability_png',
    'source_concentration_png',
    'climate_response_management_network_png',
    'evidence_synthesis_report_md',
    'methods_and_results_md',
    'executive_decision_brief_md',
]
manifest = build_product_manifest(ROOT, paths, product_keys)
missing = manifest.loc[~manifest['exists'].astype(bool)]
empty = manifest.loc[manifest['size_bytes'].astype(int).le(0)]
manifest_issues = []
for row in missing.to_dict('records'):
    manifest_issues.append({
        'output': 'product_manifest',
        'row_number': 1,
        'record_id': row['product_key'],
        'field': 'relative_path',
        'issue': 'configured_product_missing',
    })
for row in empty.to_dict('records'):
    manifest_issues.append({
        'output': 'product_manifest',
        'row_number': 1,
        'record_id': row['product_key'],
        'field': 'size_bytes',
        'issue': 'configured_product_empty',
    })
if manifest_issues:
    issues = pd.concat([issues, pd.DataFrame(manifest_issues)], ignore_index=True)

manifest_path = ROOT / paths['product_manifest_csv']
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(manifest_path, index=False, encoding='utf-8-sig')
issues_path = ROOT / paths['publication_issues_csv']
issues.to_csv(issues_path, index=False, encoding='utf-8-sig')

print(f'Configured products: {len(manifest)}')
print(f'Missing products: {len(missing)}')
print(f'Empty products: {len(empty)}')
print(f'Final publication issues: {len(issues)}')
display(manifest)
if not issues.empty:
    raise ValueError('Phase 13 product validation failed.')


## Criterio de cierre

La fase 13 queda cerrada cuando todos los productos configurados existen, `Final publication issues = 0`, cada recomendación validada aparece una vez en la tabla final y ninguna opción se presenta como efectiva sin evaluación de resultados. Los productos se generan bajo `data/processed/phase13/` y permanecen fuera de Git.
